In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize
import nltk

# Download nltk tokenizer
nltk.download('punkt_tab')

# Define risk-related keywords
RISK_KEYWORDS = [
    "risk", "uncertainty", "exposure", "threat", "liability",
    "vulnerability", "instability", "adverse effect", "fluctuation",
    "deterioration", "litigation", "compliance", "regulatory",
    "economic downturn", "competition", "market volatility",
    "legal action", "fraud", "operational risk", "data breach",
    "cybersecurity", "lawsuit", "supply chain disruption",
    "financial loss", "inflation", "environmental impact",
    "government regulation", "sanctions"
]

def extract_risk_sentences(file_path, window=2):
    """
    Extract sentences containing risk-related keywords along with surrounding context.

    Parameters:
        file_path (str): Path to the 10-K text file.
        window (int): Number of surrounding sentences to include for context.

    Returns:
        str: Extracted risk-related sentences with context.
    """
    try:
        # Read the file content
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()

        # Remove HTML tags using BeautifulSoup
        soup = BeautifulSoup(content, 'html.parser')
        text = soup.get_text()

        # Normalize spaces and clean text
        text = re.sub(r'\s+', ' ', text)  # Normalize spaces
        text = re.sub(r'[^a-zA-Z0-9\s\.\,\-]', '', text)  # Remove non-text characters

        # Split text into sentences
        sentences = sent_tokenize(text)

        # Find risk-related sentences with context
        extracted_sentences = []
        for i, sentence in enumerate(sentences):
            if any(re.search(rf"\b{kw}\b", sentence, re.IGNORECASE) for kw in RISK_KEYWORDS):
                # Extract surrounding sentences for better context
                start = max(0, i - window)
                end = min(len(sentences), i + window + 1)
                context = " ".join(sentences[start:end])
                extracted_sentences.append(context)

        return "\n\n".join(extracted_sentences) if extracted_sentences else "No relevant risk-related sentences found."

    except Exception as e:
        return f"Error processing file: {e}"

def process_all_10k_filings(base_folder):
    """
    Recursively traverse subfolders within the SEC filings directory,
    process all '10-K' reports, and extract risk-related sentences.
    Writes the output into 'refined.txt' in the same directory.
    """
    for root, dirs, files in os.walk(base_folder):
        for file_name in files:
            if file_name == "full-submission.txt":  # Process 'full-submission.txt' files
                file_path = os.path.join(root, file_name)
                refined_txt_path = os.path.join(root, "risk_refined.txt")

                # Ensure 'refined.txt' exists before writing
                if not os.path.exists(refined_txt_path):
                    print(f"Skipping {root}: 'refined.txt' not found.")
                    continue

                print(f"Processing: {file_path}")

                # Extract Risk Sentences
                risk_sentences = extract_risk_sentences(file_path)

                # Append to refined.txt
                with open(refined_txt_path, 'a', encoding='utf-8') as output_file:
                    output_file.write("\n\n" + risk_sentences)

                print(f"Appended risk sentences to: {refined_txt_path}")

# Example Usage
base_folder = "/content/drive/MyDrive/Assignment_1/data files/sec-edgar-filings"
process_all_10k_filings(base_folder)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Processing: /content/drive/MyDrive/Assignment_1/data files/sec-edgar-filings/0000006201/10-K/0000006201-24-000010/full-submission.txt
Appended risk sentences to: /content/drive/MyDrive/Assignment_1/data files/sec-edgar-filings/0000006201/10-K/0000006201-24-000010/risk_refined.txt
Processing: /content/drive/MyDrive/Assignment_1/data files/sec-edgar-filings/0000006201/10-K/0000006201-23-000018/full-submission.txt
Appended risk sentences to: /content/drive/MyDrive/Assignment_1/data files/sec-edgar-filings/0000006201/10-K/0000006201-23-000018/risk_refined.txt
Processing: /content/drive/MyDrive/Assignment_1/data files/sec-edgar-filings/0000006201/10-K/0000006201-22-000026/full-submission.txt
Appended risk sentences to: /content/drive/MyDrive/Assignment_1/data files/sec-edgar-filings/0000006201/10-K/0000006201-22-000026/risk_refined.txt
Processing: /content/drive/MyDrive/Assignment_1/data files/sec-edgar-filings/0000006201/10-K/0000006201-21-000014/full-submission.txt
Appended risk sentences